In [1]:
from pathlib import Path
import sys

ROOT_DIR = next(
    (candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "src").exists()),
    None,
)

if ROOT_DIR is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import PORTAL_TRANSPARENCIA_API_KEY, RAW_DIR, INTERIM_DIR

In [2]:
import pandas as pd
from pathlib import Path

# Ajuste para um dos seus arquivos
arquivo = RAW_DIR / "202201_Licitacoes" / "202201_Licitacao.csv"

# Os dados do governo geralmente vêm em latin-1 com ;
df_test = pd.read_csv(arquivo, sep=";", encoding="latin-1", nrows=5)
print(df_test.columns.tolist())
df_test.head()

# Exibe o diretório de trabalho para depuração
print("CWD:", Path.cwd())
print("ROOT_DIR:", ROOT_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

# Leitura das estruturas de dados (schemas)
df_licitacao = pd.read_parquet(INTERIM_DIR / "licitacao.parquet")
df_participantes = pd.read_parquet(INTERIM_DIR / "participanteslicitacao.parquet")

# Exibição dos rótulos de colunas para identificação de chaves estruturais
print("Campos - Licitação:", df_licitacao.columns.tolist())
print("Campos - Participantes:", df_participantes.columns.tolist())

['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Objeto', 'Situação Licitação', 'Código Órgão Superior', 'Nome Órgão Superior', 'Código Órgão', 'Nome Órgão', 'UF', 'Município', 'Data Resultado Compra', 'Data Abertura', 'Valor Licitação']
CWD: c:\Users\Gabriela\Pmonkey\IA\predicao\notebooks
ROOT_DIR: C:\Users\Gabriela\Pmonkey\IA\predicao
INTERIM_DIR: C:\Users\Gabriela\Pmonkey\IA\predicao\data\interim
Campos - Licitação: ['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Objeto', 'Situação Licitação', 'Código Órgão Superior', 'Nome Órgão Superior', 'Código Órgão', 'Nome Órgão', 'UF', 'Município', 'Data Resultado Compra', 'Data Abertura', 'Valor Licitação', 'ano_mes_arquivo']
Campos - Participantes: ['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Código Órgão', 'Nome Órgão', 'Código Item Compra', '

In [3]:
import sys
from pathlib import Path
import pandas as pd

# Alinha o path do sistema para reconhecer o pacote 'src' a partir do diretório de notebooks
sys.path.append(str(Path.cwd().parent))
from src.config import INTERIM_DIR

# Carregamento dos dados analíticos
df_licitacao = pd.read_parquet(INTERIM_DIR / "licitacao.parquet")

# Definição da chave de entidade
chave_composta = ['Número Licitação', 'Código UG']

# Avaliação de duplicidade estrutural
total_registros = len(df_licitacao)
total_duplicados = df_licitacao.duplicated(subset=chave_composta).sum()

print(f"Volume total de registros analisados: {total_registros:,}")
print(f"Registros violando a restrição de unicidade: {total_duplicados:,}")

if total_duplicados > 0:
    print("\nExemplo de registros duplicados detectados (amostra):")
    df_duplicados = df_licitacao[df_licitacao.duplicated(subset=chave_composta, keep=False)]
    print(df_duplicados[chave_composta + ['ano_mes_arquivo']].sort_values(by=chave_composta).head(4))

Volume total de registros analisados: 109,346
Registros violando a restrição de unicidade: 21,517

Exemplo de registros duplicados detectados (amostra):
      Número Licitação Código UG ano_mes_arquivo
6            000012021    160220          202201
25938        000012021    160220          202205
18252        000012021    160295          202204
36993        000012021    160295          202206
